# 8 Ball Table Analyses - Task 1 Computer Vision

## Imports

In [ ]:
import os
import cv2
import matplotlib.pyplot as plt
import numpy as np
import math

## Load Dataset

In [ ]:
DATASET_DIR = "development_set/"

In [ ]:
image_paths = sorted([
    os.path.join(DATASET_DIR, f)
    for f in os.listdir(DATASET_DIR)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
])

print(f"Found {len(image_paths)} images")

In [ ]:
def show_images(images, titles=None, max_cols=4, figsize_per_image=(4, 3)):
    n = len(images)
    if n == 0:
        print("No images to display.")
        return

    cols = min(n, max_cols)
    rows = (n + cols - 1) // cols

    fig, axes = plt.subplots(rows, cols, figsize=(figsize_per_image[0] * cols,
                                                   figsize_per_image[1] * rows))
    axes = np.array(axes).flatten()

    for i, ax in enumerate(axes):
        if i < n:
            img = images[i]
            if isinstance(img, str):
                img = cv2.imread(img)

            if img is not None:
                if img.ndim == 2:                          # ← grayscale / mask
                    ax.imshow(img, cmap='gray', vmin=0, vmax=255)
                else:                                      # ← colour image
                    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

            ax.set_title(titles[i] if titles and i < len(titles) else f"Image {i+1}",
                         fontsize=9)
        ax.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
show_images(image_paths, titles=[os.path.basename(p) for p in image_paths])

## Extract Table

In [ ]:
def preprocess_lighting(hsv_image):
    # 1. Split the HSV image into its three separate channels
    h, s, v = cv2.split(hsv_image)

    # 2. Create the CLAHE filter
    # clipLimit prevents noise from being amplified too much
    # tileGridSize is the size of the localized "checkerboard" squares
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

    # 3. Apply the filter ONLY to the V (brightness) channel
    v_eq = clahe.apply(v)

    # 4. Merge the channels back together
    hsv_eq = cv2.merge((h, s, v_eq))

    return hsv_eq

In [ ]:
def get_table_mask(image):
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    hsv = preprocess_lighting(hsv)
    h, w = image.shape[:2]

    # Sample using median to avoid balls in the center
    # 1. Define 5 safe sampling points (Center + 4 inner quadrants)
    points = [
        (h//2, w//2),               # Center
        (int(h*0.4), int(w*0.4)),   # Top-Left inner
        (int(h*0.4), int(w*0.6)),   # Top-Right inner
        (int(h*0.6), int(w*0.4)),   # Bottom-Left inner
        (int(h*0.6), int(w*0.6))    # Bottom-Right inner
    ]

    # 2. Collect a 40x40 patch from ALL 5 locations
    samples = []
    for py, px in points:
        patch = hsv[py-20:py+20, px-20:px+20]
        samples.append(patch)

    # 3. Stack all 8,000 pixels together and find the true median
    all_samples = np.vstack(samples)
    median_hsv = np.median(all_samples, axis=(0, 1))

    # Build tolerance and threshold
    tol = np.array([15, 80, 80])
    lower = np.clip(median_hsv - tol, 0, 255).astype(np.uint8)
    upper = np.clip(median_hsv + tol, 0, 255).astype(np.uint8)

    return cv2.inRange(hsv, lower, upper)

In [ ]:
images = [cv2.imread(p) for p in image_paths]
masks  = [get_table_mask(img) for img in images]

show_images(masks, titles=[os.path.basename(p) for p in image_paths])

In [ ]:
def isolate_largest_blob(mask):
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if not contours:
        return np.zeros_like(mask)

    largest_contour = max(contours, key=cv2.contourArea)

    clean_mask = np.zeros_like(mask)

    cv2.drawContours(clean_mask, [largest_contour], -1, 255, thickness=cv2.FILLED)

    return clean_mask

In [ ]:
def show_images_with_masks(images, masks, titles=None, figsize_per_image=(4, 3)):
    n = len(images)

    # 2 pairs per row = 4 total columns (Image 1, Mask 1, Image 2, Mask 2)
    pairs_per_row = 2
    cols = pairs_per_row * 2
    rows = math.ceil(n / pairs_per_row)

    # Create the grid
    fig, axes = plt.subplots(rows, cols, figsize=(figsize_per_image[0] * cols,
                                                   figsize_per_image[1] * rows))

    # Flatten the 2D axes array into a simple 1D list so we can loop through it easily
    if rows == 1 and cols == 1:
        axes = [axes] # Edge case handler
    else:
        axes = axes.flatten()

    for i, (img, mask) in enumerate(zip(images, masks)):
        if isinstance(img, str):
            img = cv2.imread(img)

        title = titles[i] if titles and i < len(titles) else f"Image {i+1}"

        # Calculate where this pair should be placed in the flattened array
        idx = i * 2

        # 1. Draw Original Image
        axes[idx].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        axes[idx].set_title(title, fontsize=9)
        axes[idx].axis("off")

        # 2. Draw Mask right next to it
        axes[idx + 1].imshow(mask, cmap='gray', vmin=0, vmax=255)
        axes[idx + 1].set_title(f"{title} - mask", fontsize=9)
        axes[idx + 1].axis("off")

    # Clean up: Hide any leftover empty subplots if 'n' is an odd number
    for j in range(n * 2, rows * cols):
        axes[j].axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
imgs = [cv2.imread(p) for p in image_paths]

raw_masks = [get_table_mask(img) for img in imgs]

clean_masks = [isolate_largest_blob(mask) for mask in raw_masks]

show_images_with_masks(imgs, clean_masks, titles=[os.path.basename(p) for p in image_paths])

## Get Top view

In [ ]:
def order_points(pts):
    """Sorts the 4 corners into a consistent order: TL, TR, BR, BL"""
    rect = np.zeros((4, 2), dtype="float32")

    # Top-Left has the smallest sum (x+y), Bottom-Right has the largest sum
    s = pts.sum(axis=1)
    rect[0] = pts[np.argmin(s)]
    rect[2] = pts[np.argmax(s)]

    # Top-Right has the smallest difference (y-x), Bottom-Left has the largest difference
    diff = np.diff(pts, axis=1)
    rect[1] = pts[np.argmin(diff)]
    rect[3] = pts[np.argmax(diff)]

    return rect

In [ ]:
def get_table_corners(clean_mask):
    contours, _ = cv2.findContours(clean_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None

    largest_contour = max(contours, key=cv2.contourArea)
    hull = cv2.convexHull(largest_contour)

    epsilon = 0.02 * cv2.arcLength(hull, True)
    corners = cv2.approxPolyDP(hull, epsilon, True)

    if len(corners) == 4:
        return corners
    else:
        print(f"Warning: Found {len(corners)} corners instead of 4.")
        return None

In [ ]:
corner_debug_images = []

for img in imgs:
    debug_img = img.copy()

    raw_mask = get_table_mask(debug_img)
    clean_mask = isolate_largest_blob(raw_mask)
    corners = get_table_corners(clean_mask)

    if corners is not None and len(corners) == 4:
        for point in corners:
            x, y = point[0]
            cv2.circle(debug_img, (x, y), 15, (0, 0, 255), -1)
    else:
        print("Warning: Did not find 4 corners on one of the images.")

    corner_debug_images.append(debug_img)

show_images(corner_debug_images, titles=[os.path.basename(p) for p in image_paths])

In [ ]:
def warp_to_top_view(original_image, corners):
    # Safety check: Don't try to warp if we didn't find exactly 4 corners
    if corners is None or len(corners) != 4:
        return None

    # 1. Flatten the OpenCV corner array format
    pts = corners.reshape(4, 2)

    # 2. Sort them (Top-Left, Top-Right, Bottom-Right, Bottom-Left)
    rect = order_points(pts)

    # 3. Define the size of the new Top-View (2:1 Pool Table Ratio)
    width = 1000
    height = 500

    dst = np.array([
        [0, 0],
        [width - 1, 0],
        [width - 1, height - 1],
        [0, height - 1]
    ], dtype="float32")

    # 4. Do the math and warp the image
    matrix = cv2.getPerspectiveTransform(rect, dst)
    warped_image = cv2.warpPerspective(original_image, matrix, (width, height))

    return warped_image